# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
%pip -q install duckdb huggingface_hub

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF Token loaded successfully!")

HF Token loaded successfully!


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connected successfully!")

DuckDB connected successfully!


In [5]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [6]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


Unit of analysis:
One row represents one content item (content_hash_id) for one client (client_hash_id).

Time window:
This analysis uses a mid-panel month (March 2026) to avoid data leakage from the final month.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## Features
- imp_prev30
- visible_queries
- rare_share
- anon_share
- top_query_share

## Label
- is_declining

## Context
- client_hash_id
- content_hash_id
- report_date

## Excluded
- imp_last30

Reason:
imp_last30 is excluded because it is used to create the label. Including it as a feature would cause data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = {
    "Feature": [
        "imp_prev30",
        "visible_queries",
        "rare_share",
        "anon_share",
        "top_query_share"
    ],
    "Label": [
        "is_declining"
    ],
    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ],
    "Excluded": [
        "imp_last30"
    ]
}

for category, values in fields.items():
    print(f"{category}:")
    for value in values:
        print(" -", value)
    print()


Feature:
 - imp_prev30
 - visible_queries
 - rare_share
 - anon_share
 - top_query_share

Label:
 - is_declining

Context:
 - client_hash_id
 - content_hash_id
 - report_date

Excluded:
 - imp_last30



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR)) AS unique_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows,start_date,end_date
0,9841378,9841378,2026-03-01,2026-03-31


In [12]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 1
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01


In [13]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


In [14]:
con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,content_items
0,55,331437


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Data limitations:

- Some clients have less historical data than others.
- Early records may contain only Google Search Console data.
- This analysis uses only March 2026, so results may not generalize to all months.
- The dataset cannot explain why rankings changed (algorithm updates or competitor changes are not included).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.